In [ ]:
import os
import csv
import ast
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import matplotlib as mpl
import sys
sys.path.insert(0, str(Path("..").resolve()))
from utils import savefig

plt.rcParams['font.size'] = 14
mpl.rcParams['svg.fonttype'] = 'none'

save_fig = False

In [ ]:
gamma_names = ["0", "02", "04", "06", "08", "10"]
temporal_discount_factors = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
noise_levels = [0, 0.2, 0.4, 0.6, 0.8, 1]
noise_names = ["0", "02", "04", "06", "08", "1"]
colormap = np.array(["#2A9D8F", "#E9C46A", "#E76F51", "#F4A261", "#264653"]) 

In [ ]:
df = pd.read_pickle("../data/df_models.pkl")
df_filtered = pd.read_pickle("../data/df_filtered_cluster.pkl")
df_reservoir = pd.read_pickle("../data/df_reservoir.pkl")
df_tcm = pd.read_pickle("../data/df_tcm.pkl")
df_perf = pd.read_pickle("../data/df_perf.pkl")

### plot relationship of metrics with clusters

In [ ]:
# plot scatter with cluster labels

def plot_scatter_cluster(df, x, y, c, x_label, y_label, fit=True, plot_reservoir=True, save_folder=None, save_name=None):
    plt.figure(figsize=(3.6, 3.3), dpi=200)

    plt.scatter(df[x], df[y], c=colormap[df[c].iloc[:]], alpha=0.5)

    if fit:
        if plot_reservoir:
            # plot for reservoir and tcm
            plt.scatter(df_reservoir[x], df_reservoir[y], c=colormap[-1], alpha=0.5)
            # plt.scatter(df_tcm[x], df_tcm[y], c=colormap[-1])
    
        # Reshape x and y for sklearn
        x_reshaped = np.array(df[x]).reshape(-1, 1)
        y_reshaped = np.array(df[y]).reshape(-1, 1)

        # Fit linear regression model
        model = LinearRegression()
        model.fit(x_reshaped, y_reshaped)

        # Predict y values
        sorted_x_idx = np.argsort(x_reshaped.reshape(-1))
        x1, x2 = x_reshaped[sorted_x_idx[0]], x_reshaped[sorted_x_idx[-1]]
        y_pred = model.predict(x_reshaped)
        y1, y2 = y_pred[sorted_x_idx[0]], y_pred[sorted_x_idx[-1]]

        # Calculate R-squared
        r2 = r2_score(y_reshaped, y_pred)
        # Calculate the p-value for the regression
        r, p_value = stats.pearsonr(x_reshaped.flatten(), y_reshaped.flatten())
        
        # Plot the regression line
        plt.plot([x1, x2], [y1, y2], color="grey")

        x_label = x_label # +"\n(R² = {:.2f}, p = {:.3f})".format(r2, p_value)
        print("freedom:", x_reshaped.shape[0]-2, "r2: ", r2, "r: ", r, "p_value: ", p_value)

        ax = plt.gca()
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    plt.xlabel(x_label)
    plt.ylabel(y_label)

    plt.tight_layout()

    if save_fig and save_folder and save_name:
        savefig(save_folder, save_name, format="svg", close=False)

    # plt.tight_layout()
    plt.show()


In [ ]:
plot_scatter_cluster(df_filtered, 'forward_asymmetry', 'temporal_factor', 'cluster', 'Forward asymmetry', 'Temporal organization score',
                    save_folder="./figures/fig2", save_name="fig2B")
plot_scatter_cluster(df_filtered, 'explained_variance_index', 'explained_variance_identity', 'cluster', 'Variance explained\n(index)', 'Variance explained\n(identity)',
                    save_folder="./figures/fig2", save_name="fig2C")
plot_scatter_cluster(df_filtered, 'cross_decoding_accuracy_index', 'cross_decoding_accuracy_identity', 'cluster', 'Cross decoding accuracy\n(index)', 'Cross decoding accuracy\n(identity)',
                    save_folder="./figures/supp3", save_name="suppfig3C")

plot_scatter_cluster(df_filtered, 'forward_asymmetry', 'explained_variance_index', 'cluster', 'Forward asymmetry', 'Variance explained (index)',
                    save_folder="./figures/fig3", save_name="fig3D")
plot_scatter_cluster(df_filtered, 'forward_asymmetry', 'cross_decoding_accuracy_index', 'cluster', 'Forward asymmetry', 'Cross decoding accuracy\n(index)',
                    save_folder="./figures/supp3", save_name="suppfig3D")

plot_scatter_cluster(df_filtered, 'temporal_factor', 'explained_variance_index', 'cluster', 'Temporal organization score', 'Variance explained (index)',
                    save_folder="./figures/fig3", save_name="fig3E")
plot_scatter_cluster(df_filtered, 'temporal_factor', 'cross_decoding_accuracy_index', 'cluster', 'Temporal organization score', 'Cross decoding accuracy\n(index)',
                    save_folder="./figures/supp3", save_name="suppfig3G")


In [ ]:
plot_scatter_cluster(df_filtered, 'forward_asymmetry', 'explained_variance_identity', 'cluster', 'Forward asymmetry', 'Variance explained\n(identity)',
                    save_folder="./figures/supp3", save_name="suppfig3F")
plot_scatter_cluster(df_filtered, 'forward_asymmetry', 'cross_decoding_accuracy_identity', 'cluster', 'Forward asymmetry', 'Cross decoding accuracy\n(identity)',
                    save_folder="./figures/supp3", save_name="suppfig3E")

plot_scatter_cluster(df_filtered, 'temporal_factor', 'explained_variance_identity', 'cluster', 'Temporal organization score', 'Variance explained\n(identity)',
                    save_folder="./figures/supp3", save_name="suppfig3I")
plot_scatter_cluster(df_filtered, 'temporal_factor', 'cross_decoding_accuracy_identity', 'cluster', 'Temporal organization score', 'Cross decoding accuracy\n(identity)',
                    save_folder="./figures/supp3", save_name="suppfig3H")

In [ ]:
# Create a legend for the colormap
import matplotlib.patches as mpatches

# Define labels for the legend
legend_labels = ['Cluster 1', 'Cluster 2', 'Cluster 3', 'Cluster 4', 'Cluster 5']

# Create a list of patches for the legend
patches = [mpatches.Patch(color="#2A9D8F", label="Strategy 1: Memory palace"),
           mpatches.Patch(color="#E76F51", label="Strategy 2: TCM-like forward"),
           mpatches.Patch(color="#E9C46A", label="Strategy 3: TCM-like backward"),
           mpatches.Patch(color="#264653", label="Reservoir RNN")]
        #    mpatches.Patch(color="#264653", label="TCM")]
# [mpatches.Patch(color=colormap[i], label=legend_labels[i]) for i in range(optimal_clusters)]

optimal_clusters = 3

# Plot the legend
plt.figure(figsize=(14, 0.5), dpi=180)
plt.legend(handles=patches, loc='center', ncol=optimal_clusters+1, frameon=False)
plt.axis('off')
# plt.tight_layout()
savefig("./figures/others", "legend_cluster1", format="svg", close=False)
plt.show()

plt.figure(figsize=(4, 1.5), dpi=180)
plt.legend(handles=patches, loc='center', frameon=False)
plt.axis('off')
# plt.tight_layout()
savefig("./figures/others", "legend_cluster2", format="svg", close=False)
plt.show()

### plot strategy distribution by hyperparameters

In [ ]:
# distribution of strategies in each factor

def plot_bar_by_strategy(df, x, x_label, save_folder=None, save_name=None):
    plt.figure(figsize=(3.2, 3.3), dpi=180)
    
    cluster_distribution = df.groupby(["cluster", x]).size().unstack(fill_value=0)
    sum_distribution = cluster_distribution.sum(axis=0)
    cluster_distribution = cluster_distribution.div(sum_distribution, axis=1)
    cumsum_distribution = cluster_distribution.cumsum(axis=0)

    for i, row in cumsum_distribution.iterrows():
        plt.bar(row.index, row.values, color=colormap[i], label=f"Cluster {i}", width=0.12, zorder=-i)
    plt.xlabel(x_label)
    plt.ylabel("Distribution of strategies")

    unique_x = df[x].unique()
    plt.xticks(unique_x, unique_x)
    
    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()

    if save_fig and save_folder and save_name:
        savefig(save_folder, save_name, format="svg", close=False)

    plt.show()


In [ ]:
df_all_tdf = df_filtered[(df_filtered["noise_level"] == 1.0)]

plot_bar_by_strategy(df_all_tdf, "temporal_discount_factor", "Discount factor", save_folder="./figures/fig3", save_name="fig3F")

df_all_noise = df_filtered[df_filtered["temporal_discount_factor"] == 1.0]

plot_bar_by_strategy(df_all_noise, "noise_level", "% WM flushed", save_folder="./figures/fig3", save_name="fig3G")



### explained variance bar plot

In [ ]:
df = df_filtered

In [ ]:


mean_explained_variance_index = df.groupby("cluster")["explained_variance_index"].mean()
std_explained_variance_index = df.groupby("cluster")["explained_variance_index"].std()
mean_explained_variance_identity = df.groupby("cluster")["explained_variance_identity"].mean()
std_explained_variance_identity = df.groupby("cluster")["explained_variance_identity"].std()
mean_explained_variance_index_reservoir = df_reservoir["explained_variance_index"].mean()
std_explained_variance_index_reservoir = df_reservoir["explained_variance_index"].std()
mean_explained_variance_identity_reservoir = df_reservoir["explained_variance_identity"].mean()
std_explained_variance_identity_reservoir = df_reservoir["explained_variance_identity"].std()

mean_explained_variance_index = np.array(mean_explained_variance_index)
std_explained_variance_index = np.array(std_explained_variance_index)
mean_explained_variance_identity = np.array(mean_explained_variance_identity)
std_explained_variance_identity = np.array(std_explained_variance_identity)
mean_explained_variance_index = [mean_explained_variance_index[0], mean_explained_variance_index[2], mean_explained_variance_index[1],mean_explained_variance_index_reservoir]
std_explained_variance_index = [std_explained_variance_index[0], std_explained_variance_index[2], std_explained_variance_index[1],std_explained_variance_index_reservoir]
mean_explained_variance_identity = [mean_explained_variance_identity[0], mean_explained_variance_identity[2], mean_explained_variance_identity[1],mean_explained_variance_identity_reservoir]
std_explained_variance_identity = [std_explained_variance_identity[0], std_explained_variance_identity[2], std_explained_variance_identity[1],std_explained_variance_identity_reservoir]

plt.figure(figsize=(4.5, 3.3), dpi=180)
ax = plt.gca()
bar_width = 0.35
index = np.arange(4)
plt.bar(index, mean_explained_variance_index, bar_width, yerr=std_explained_variance_index, capsize=5, color="#A1C181")
plt.bar(index + bar_width, mean_explained_variance_identity, bar_width, yerr=std_explained_variance_identity, capsize=5, color="#62B6CB")
plt.xticks(index + bar_width / 2, ["Strategy 1", "Strategy 2", "Strategy 3", "Reservoir"], fontsize=12, rotation=30)
for xtick, color in zip(ax.get_xticklabels(), [colormap[0], colormap[2], colormap[1], colormap[-1]]):
    xtick.set_color(color)
plt.ylabel("Explained variance")
# plt.legend(["Index", "Identity"], frameon=False, fontsize=12)
for group, pos in zip([0,1,2], [0,2,1]):
    data = df[df["cluster"] == group]
    for i in range(len(data)):
        plt.plot([pos, pos+bar_width], [data["explained_variance_index"].iloc[i], data["explained_variance_identity"].iloc[i]], color='grey', linestyle='-', linewidth=0.3, alpha=0.4)
for i in range(len(df_reservoir)):
    plt.plot([3, 3+bar_width], [df_reservoir["explained_variance_index"].iloc[i], df_reservoir["explained_variance_identity"].iloc[i]], color='grey', linestyle='-', linewidth=0.3, alpha=0.4)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

import matplotlib.patches as mpatches
patches = [mpatches.Patch(color="#A1C181", label="Index"), mpatches.Patch(color="#62B6CB", label="Identity")]
plt.legend(handles=patches, frameon=False, fontsize=12)

plt.tight_layout()
if save_fig:
    savefig("./figures/fig3", "fig3A", format="svg", close=False)
plt.show()


In [ ]:
# mean cross decoding accuracy of each strategy

mean_cross_decoding_accuracy_index = df.groupby("cluster")["cross_decoding_accuracy_index"].mean()
std_cross_decoding_accuracy_index = df.groupby("cluster")["cross_decoding_accuracy_index"].std()
mean_cross_decoding_accuracy_identity = df.groupby("cluster")["cross_decoding_accuracy_identity"].mean()
std_cross_decoding_accuracy_identity = df.groupby("cluster")["cross_decoding_accuracy_identity"].std()
mean_cross_decoding_accuracy_index_reservoir = df_reservoir["cross_decoding_accuracy_index"].mean()
std_cross_decoding_accuracy_index_reservoir = df_reservoir["cross_decoding_accuracy_index"].std()
mean_cross_decoding_accuracy_identity_reservoir = df_reservoir["cross_decoding_accuracy_identity"].mean()
std_cross_decoding_accuracy_identity_reservoir = df_reservoir["cross_decoding_accuracy_identity"].std()

mean_cross_decoding_accuracy_index = np.array(mean_cross_decoding_accuracy_index)
std_cross_decoding_accuracy_index = np.array(std_cross_decoding_accuracy_index)
mean_cross_decoding_accuracy_identity = np.array(mean_cross_decoding_accuracy_identity)
std_cross_decoding_accuracy_identity = np.array(std_cross_decoding_accuracy_identity)
mean_cross_decoding_accuracy_index = [mean_cross_decoding_accuracy_index[0], mean_cross_decoding_accuracy_index[2], mean_cross_decoding_accuracy_index[1],mean_cross_decoding_accuracy_index_reservoir]
std_cross_decoding_accuracy_index = [std_cross_decoding_accuracy_index[0], std_cross_decoding_accuracy_index[2], std_cross_decoding_accuracy_index[1],std_cross_decoding_accuracy_index_reservoir]
mean_cross_decoding_accuracy_identity = [mean_cross_decoding_accuracy_identity[0], mean_cross_decoding_accuracy_identity[2], mean_cross_decoding_accuracy_identity[1],mean_cross_decoding_accuracy_identity_reservoir]
std_cross_decoding_accuracy_identity = [std_cross_decoding_accuracy_identity[0], std_cross_decoding_accuracy_identity[2], std_cross_decoding_accuracy_identity[1],std_cross_decoding_accuracy_identity_reservoir]

plt.figure(figsize=(4.5, 3.3), dpi=180)
ax = plt.gca()
bar_width = 0.35
index = np.arange(4)
plt.bar(index, mean_cross_decoding_accuracy_index, bar_width, yerr=std_cross_decoding_accuracy_index, capsize=5, color="#A1C181")
plt.bar(index + bar_width, mean_cross_decoding_accuracy_identity, bar_width, yerr=std_cross_decoding_accuracy_identity, capsize=5, color="#62B6CB")
plt.xticks(index + bar_width / 2, ["Strategy 1", "Strategy 2", "Strategy 3", "Reservoir"], fontsize=12, rotation=30)
for xtick, color in zip(ax.get_xticklabels(), [colormap[0], colormap[2], colormap[1], colormap[-1]]):
    xtick.set_color(color)
plt.ylabel("cross-phase\ndecoding accuracy")
# plt.legend(["Index", "Identity"], frameon=False, fontsize=12)
for group, pos in zip([0,1,2], [0, 2, 1]):
    data = df[df["cluster"] == group]
    for i in range(len(data)):
        plt.plot([pos, pos+bar_width], [data["cross_decoding_accuracy_index"].iloc[i], data["cross_decoding_accuracy_identity"].iloc[i]], color='grey', linestyle='-', linewidth=0.3, alpha=0.5)
for i in range(len(df_reservoir)):
    plt.plot([3, 3+bar_width], [df_reservoir["cross_decoding_accuracy_index"].iloc[i], df_reservoir["cross_decoding_accuracy_identity"].iloc[i]], color='grey', linestyle='-', linewidth=0.3, alpha=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.show()


In [ ]:
# mean item invariance accuracy of each strategy

item_invariance_accuracy_mean = []
item_invariance_accuracy_std = []
for i in range(3):
    item_invariance_data = df[df["cluster"] == i]["item_invariance_accuracy"]
    item_invariance_data = np.stack(item_invariance_data)
    item_invariance_accuracy_mean.append(np.mean(item_invariance_data, axis=0))
    item_invariance_accuracy_std.append(np.std(item_invariance_data, axis=0))

item_invariance_accuracy_mean_reservoir = df_reservoir["item_invariance_accuracy"].mean()
item_invariance_accuracy_std_reservoir = np.std(np.array(df_reservoir["item_invariance_accuracy"]))

plt.figure(figsize=(3.7, 3.3), dpi=180)
plt.plot(item_invariance_accuracy_mean[0][::-1], color=colormap[0])
plt.errorbar(range(len(item_invariance_accuracy_mean[0])), item_invariance_accuracy_mean[0][::-1], yerr=item_invariance_accuracy_std[0][::-1], fmt='o', color=colormap[0], capsize=3)
plt.plot(item_invariance_accuracy_mean[1][::-1], color=colormap[1])
plt.errorbar(range(len(item_invariance_accuracy_mean[1])), item_invariance_accuracy_mean[1][::-1], yerr=item_invariance_accuracy_std[1][::-1], fmt='o', color=colormap[1], capsize=3)
plt.plot(item_invariance_accuracy_mean[2][::-1], color=colormap[2])
plt.errorbar(range(len(item_invariance_accuracy_mean[2])), item_invariance_accuracy_mean[2][::-1], yerr=item_invariance_accuracy_std[2][::-1], fmt='o', color=colormap[2], capsize=3)
plt.plot(item_invariance_accuracy_mean_reservoir[::-1], color=colormap[-1])
plt.errorbar(range(len(item_invariance_accuracy_mean_reservoir)), item_invariance_accuracy_mean_reservoir[::-1], yerr=item_invariance_accuracy_std_reservoir, fmt='o', color=colormap[-1], capsize=3)
for i in range(3):
    item_invariance_data = df[df["cluster"] == i]["item_invariance_accuracy"]
    item_invariance_data = np.stack(item_invariance_data)
    for j in range(len(item_invariance_data)):
        plt.plot(item_invariance_data[j][::-1], color=colormap[i], linestyle='-', linewidth=0.3, alpha=0.3)
for j in range(len(df_reservoir)):
    plt.plot(df_reservoir["item_invariance_accuracy"].iloc[j][::-1], color=colormap[-1], linestyle='-', linewidth=0.3, alpha=0.3)
plt.xlabel("Number of items for training")
plt.ylabel("decoding accuracy of index")
ax = plt.gca()
ax.set_xticks(range(4), [64, 32, 16, 8])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.show()


In [ ]:
# mean item invariance accuracy of each strategy

time_invariance_accuracy_mean = []
time_invariance_accuracy_std = []
for i in range(3):
    time_invariance_data = df[df["cluster"] == i]["time_invariance_accuracy"]
    time_invariance_data = np.stack(time_invariance_data)
    time_invariance_accuracy_mean.append(np.mean(time_invariance_data, axis=0))
    time_invariance_accuracy_std.append(np.std(time_invariance_data, axis=0))

time_invariance_accuracy_mean_reservoir = df_reservoir["time_invariance_accuracy"].mean()
time_invariance_accuracy_std_reservoir = np.std(np.array(df_reservoir["time_invariance_accuracy"]))

plt.figure(figsize=(3.7, 3.3), dpi=180)
plt.plot(time_invariance_accuracy_mean[0][::-1], color=colormap[0])
plt.errorbar(range(len(time_invariance_accuracy_mean[0])), time_invariance_accuracy_mean[0][::-1], yerr=time_invariance_accuracy_std[0][::-1], fmt='o', color=colormap[0], capsize=3)
plt.plot(time_invariance_accuracy_mean[1][::-1], color=colormap[1])
plt.errorbar(range(len(time_invariance_accuracy_mean[1])), time_invariance_accuracy_mean[1][::-1], yerr=time_invariance_accuracy_std[1][::-1], fmt='o', color=colormap[1], capsize=3)
plt.plot(time_invariance_accuracy_mean[2][::-1], color=colormap[2])
plt.errorbar(range(len(time_invariance_accuracy_mean[2])), time_invariance_accuracy_mean[2][::-1], yerr=time_invariance_accuracy_std[2][::-1], fmt='o', color=colormap[2], capsize=3)
plt.plot(time_invariance_accuracy_mean_reservoir[::-1], color=colormap[-1])
plt.errorbar(range(len(time_invariance_accuracy_mean_reservoir)), time_invariance_accuracy_mean_reservoir[::-1], yerr=time_invariance_accuracy_std_reservoir[::-1], fmt='o', color=colormap[-1], capsize=3)
for i in range(3):
    time_invariance_data = df[df["cluster"] == i]["time_invariance_accuracy"]
    time_invariance_data = np.stack(time_invariance_data)
    for j in range(len(time_invariance_data)):
        plt.plot(time_invariance_data[j][::-1], color=colormap[i], linestyle='-', linewidth=0.3, alpha=0.3)
for j in range(len(df_reservoir)):
    plt.plot(df_reservoir["time_invariance_accuracy"].iloc[j][::-1], color=colormap[-1], linestyle='-', linewidth=0.3, alpha=0.3)
plt.xlabel("Number of time steps\nfor training")
plt.ylabel("decoding accuracy\nof identity")
ax = plt.gca()
ax.set_xticks(range(4), [8, 4, 2, 1])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.show()


In [ ]:
item_invariance_accuracy_mean_all = np.array([item_invariance_accuracy_mean[0][0], item_invariance_accuracy_mean[1][0], item_invariance_accuracy_mean[2][0], item_invariance_accuracy_mean_reservoir[0]])
item_invariance_accuracy_std_all = np.array([item_invariance_accuracy_std[0][0], item_invariance_accuracy_std[1][0], item_invariance_accuracy_std[2][0], item_invariance_accuracy_std_reservoir[0]])
item_invariance_data_clustered = []
for i in [0,2,1]:
    item_invariance_data = df[df["cluster"] == i]["item_invariance_accuracy"]
    item_invariance_data = np.stack(item_invariance_data)
    item_invariance_data_clustered.append(item_invariance_data[:, 0])
item_invariance_data_clustered.append(np.stack(df_reservoir["item_invariance_accuracy"].values)[:, 0])

plt.figure(figsize=(2.8, 3.3), dpi=180)
bar_width = 0.7
# Overlay individual datapoints for each group with jitter on the bar plot
for i, data in enumerate([item_invariance_data_clustered[0], item_invariance_data_clustered[1], item_invariance_data_clustered[2], item_invariance_data_clustered[3]]):
    x_jitter = np.random.uniform(-bar_width/3, bar_width/3, size=len(data))
    plt.scatter(np.full_like(data, [0,1,2,3][i]) + x_jitter, data, color="grey", alpha=0.3, s=7, edgecolor="none", zorder=3)
# Draw bar plot WITHOUT errorbars first (so bar and scatter are at lower zorder)
bars = plt.bar(range(4), item_invariance_accuracy_mean_all[[0, 2, 1, 3]], bar_width, color=colormap[[0, 2, 1, -1]], zorder=2)
# Overlay errorbars on top of everything (zorder high)
plt.errorbar(
    range(4),
    item_invariance_accuracy_mean_all[[0, 2, 1, 3]],
    yerr=item_invariance_accuracy_std_all[[0, 2, 1, 3]],
    fmt='none',
    ecolor='black',
    capsize=5,
    elinewidth=1.7,
    zorder=4
)

plt.xlabel("Strategies")
plt.ylabel("Cross-decoding accuracy\nof index")
ax = plt.gca()
# ax.set_xticks(range(3), ["Strategy 1", "Strategy 2", "Strategy 3"])
ax.set_xticks([])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
if save_fig:
    savefig("./figures/fig3", "fig3B", format="svg", close=False)

plt.show()

In [ ]:
time_invariance_accuracy_mean_all = np.array([time_invariance_accuracy_mean[0][-1], time_invariance_accuracy_mean[1][-1], time_invariance_accuracy_mean[2][-1], time_invariance_accuracy_mean_reservoir[-1]])
time_invariance_accuracy_std_all = np.array([time_invariance_accuracy_std[0][-1], time_invariance_accuracy_std[1][-1], time_invariance_accuracy_std[2][-1], time_invariance_accuracy_std_reservoir[-1]])
time_invariance_data_clustered = []
for i in [0,2,1]:
    time_invariance_data = df[df["cluster"] == i]["time_invariance_accuracy"]
    time_invariance_data = np.stack(time_invariance_data)
    time_invariance_data_clustered.append(time_invariance_data[:, -1])
time_invariance_data_clustered.append(np.stack(df_reservoir["time_invariance_accuracy"].values)[:, -1])

plt.figure(figsize=(2.8, 3.3), dpi=180)
bar_width = 0.7
for i, data in enumerate([time_invariance_data_clustered[0], time_invariance_data_clustered[1], time_invariance_data_clustered[2], time_invariance_data_clustered[3]]):
    x_jitter = np.random.uniform(-bar_width/3, bar_width/3, size=len(data))
    plt.scatter(np.full_like(data, [0,1,2,3][i]) + x_jitter, data, color="grey", alpha=0.3, s=6, edgecolor="none", zorder=3)
# Draw bar plot WITHOUT errorbars first (so bar and scatter are at lower zorder)
bars = plt.bar(range(4), time_invariance_accuracy_mean_all[[0, 2, 1, 3]], bar_width, color=colormap[[0, 2, 1, -1]], zorder=2)
# Overlay errorbars on top of everything (zorder high)
plt.errorbar(
    range(4),
    time_invariance_accuracy_mean_all[[0, 2, 1, 3]],
    yerr=time_invariance_accuracy_std_all[[0, 2, 1, 3]],
    fmt='none',
    ecolor='black',
    capsize=5,
    elinewidth=1.7,
    zorder=4
)

plt.xlabel("Strategies")
plt.ylabel("Cross-decoding accuracy\nof identity")
ax = plt.gca()
# ax.set_xticks(range(3), ["Strategy 1", "Strategy 2", "Strategy 3"])
ax.set_xticks([])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
if save_fig:
    savefig("./figures/fig3", "fig3C", format="svg", close=False)

plt.show()